In [1]:
import pandas as pd
import polars as pl
import svy

In [2]:
h4_df = pd.read_csv('../data/processed/h4_analytic_sample.csv')
# Drops the one record flagged as a laboratory/recording error in
# pipeline/nhanes_helpers.py (implausible_free_t3).
h4_df = h4_df[~h4_df['ft3_implausible']]
h4_pooled = h4_df.assign(wt_pooled = h4_df['wt_thyroid_2yr'] / 3)

design = svy.Design(stratum='SDMVSTRA', psu='SDMVPSU', wgt='wt_pooled')

# Primary model: excludes lipid-lowering medication users entirely
h4_primary = h4_pooled[~h4_pooled['on_lipid_lowering']]
sample_primary = svy.Sample(pl.from_pandas(h4_primary), design=design)

model_primary = svy.GLM(sample_primary)
model_primary.fit(
    y='LBXTC', 
    x=['LBXT3F', 'RIDAGEYR', svy.Cat('RIAGENDR', ref=1), 'BMXBMI'], family='gaussian')
print(model_primary)


GLM: Gaussian (identity)
  Modeling : LBXTC
  n            : 6283          DF Residuals : 45
  Deviance     : 9.2627e+06    Scale        : 191471.1880
  AIC          : 63694.3295    BIC          : -
  R-squared    : 0.10779       R-sq (adj)   : 0.10722
  F-stat (adj) : 75.77397      Prob (F-adj) : <0.001

Coefficients:
  Term          Coef.      Std.Err.  t         P>|t|   [0.025     0.975]   
  ------------  ---------  --------  --------  ------  ---------  ---------
  _intercept_   134.76816  9.68274   13.91838  <0.001  115.26611  154.27021
  LBXT3F        4.52480    2.64300   1.71199   0.0938  -0.79848   9.84807  
  RIDAGEYR      0.87206    0.04924   17.71157  <0.001  0.77289    0.97122  
  RIAGENDR_2.0  0.99694    1.66465   0.59889   0.5523  -2.35583   4.34971  
  BMXBMI        0.42085    0.11649   3.61279   <0.001  0.18623    0.65547  


C:\Users\CONNOR\AppData\Local\Temp\ipykernel_15180\1196128342.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  h4_pooled = h4_df.assign(wt_pooled = h4_df['wt_thyroid_2yr'] / 3)


In [3]:
# Sensitivity check: keeps lipid-lowering medication users and adjusts for use
h4_sensitivity = h4_pooled.assign(on_lipid_lowering=h4_pooled['on_lipid_lowering'].astype(int))
sample_sensitivity = svy.Sample(pl.from_pandas(h4_sensitivity), design=design)

model_sensitivity = svy.GLM(sample_sensitivity)
model_sensitivity.fit(
    y='LBXTC',
    x=['LBXT3F', 'RIDAGEYR', svy.Cat('RIAGENDR', ref=1), 'BMXBMI', 'on_lipid_lowering'], family='gaussian')
print(model_sensitivity)

GLM: Gaussian (identity)
  Modeling : LBXTC
  n            : 7696          DF Residuals : 44
  Deviance     : 1.1568e+07    Scale        : 239323.8458
  AIC          : 78171.0165    BIC          : -
  R-squared    : 0.10379       R-sq (adj)   : 0.10321
  F-stat (adj) : 45.77124      Prob (F-adj) : <0.001

Coefficients:
  Term               Coef.      Std.Err.  t          P>|t|   [0.025     0.975]   
  -----------------  ---------  --------  ---------  ------  ---------  ---------
  _intercept_        134.86595  8.95445   15.06134   <0.001  116.81945  152.91244
  LBXT3F             6.04232    2.51386   2.40360    0.0205  0.97597    11.10867 
  RIDAGEYR           0.74834    0.05199   14.39518   <0.001  0.64357    0.85312  
  RIAGENDR_2.0       3.92744    1.54472   2.54249    0.0146  0.81426    7.04062  
  BMXBMI             0.38091    0.11550   3.29783    0.0019  0.14813    0.61369  
  on_lipid_lowering  -33.22323  2.49291   -13.32711  <0.001  -38.24735  -28.19911
